In [11]:
import numpy as np
import math
from pyldpc import make_ldpc, encode

# 1) Desired message length and rate
k = 256
R = 1/64
n = int(k / R)             # → 16384
m = n - k                  # → 16128

# 2) Compute (dv,dc) so that (n-k)/n = dv/dc in lowest terms
g = math.gcd(m, n)
dv, dc = m//g, n//g       # → dv=63, dc=64

# 3) Build a regular (dv,dc)-LDPC in systematic form, sparse=True for speed
#    H is (m × n) sparse parity-check; G is (n × k) sparse generator
H, G = make_ldpc(n, dv, dc, systematic=True, sparse=True, seed=42)  # :contentReference[oaicite:0]{index=0}

# 4) Verify that the library gave exactly k=256 message bits
_, k_actual = G.shape
assert k_actual == k, f"Expected k={k}, got {k_actual}"

# 5) Your 256-bit message (replace with your real data)
message = np.random.randint(0, 2, size=k, dtype=int)

# 6) Encode with a very high SNR so that AWGN is negligible
#    encode(G, message, snr) does:
#      1) pure LDPC encode → binary codeword of length n
#      2) BPSK modulate (+1/–1) and add Gaussian noise
#    We then threshold at 0 to recover exact bits.
snr_db = 100.0
y = encode(G, message, snr_db)              # :contentReference[oaicite:1]{index=1}
codeword = (y < 0).astype(np.int8)           # BPSK: 1→+1, 0→–1

print("message length:", message.size)       # 256
print("codeword length:", codeword.size)     # 16384


AssertionError: Expected k=256, got 318